# S0-2 COV Characterisation — PLTS-IKN

Library-first Colab/local runner for Sprint 0 discovery. This notebook does **not** build a forecasting model, persistence baseline, production resampler, or feature pipeline. Parsing, mapping, statistics, and artifact rendering live under `src/characterisation/`.

In [ ]:
import os

REPOSITORY_URL = os.environ.get('COV_REPOSITORY_URL', 'https://github.com/ompltsikn/Forecasting-Irradiance.git')
REPO_ROOT = os.environ.get('COV_REPO_ROOT', '/content/Forecasting-Irradiance')
DRIVE_RAW_DATA_DIR = os.environ.get('COV_DRIVE_RAW_DATA_DIR', '/content/drive/Shareddrives/Forecasting Irradiance/raw_data')
LOCAL_STAGE_DIR = os.environ.get('COV_LOCAL_STAGE_DIR', '/content/cov_raw_stage')
OUTPUT_DIR = os.environ.get('COV_OUTPUT_DIR', '/content/cov_output')
REFERENCE_INVENTORY = os.environ.get('COV_REFERENCE_INVENTORY', f'{REPO_ROOT}/artifacts/phase0_cov/drive_inventory.csv')
SITE_CONFIG = os.environ.get('COV_SITE_CONFIG', f'{REPO_ROOT}/configs/site_plts-ikn.yaml')
STRICT_MODE = os.environ.get('COV_STRICT_MODE', 'true').strip().lower() in {'1', 'true', 'yes'}
SKIP_DRIVE_MOUNT = os.environ.get('COV_SKIP_DRIVE_MOUNT', 'false').strip().lower() in {'1', 'true', 'yes'}
DRIVE_OUTPUT_DIR = os.environ.get('COV_DRIVE_OUTPUT_DIR', '')

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
repo_path = Path(REPO_ROOT).resolve()
if not repo_path.is_dir():
    if not IN_COLAB:
        raise FileNotFoundError(f'Repository not found: {repo_path}')
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(repo_path)], check=True)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_path / 'requirements-cov.txt')], check=True)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print({'runtime': 'colab' if IN_COLAB else 'local', 'repo_root': str(repo_path)})

In [ ]:
if IN_COLAB and not SKIP_DRIVE_MOUNT:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Drive mount skipped; using configured filesystem path.')

In [ ]:
import hashlib
import shutil
import pandas as pd

source_dir = Path(DRIVE_RAW_DATA_DIR).resolve()
stage_dir = Path(LOCAL_STAGE_DIR).resolve()
if not source_dir.is_dir():
    raise FileNotFoundError(f'Raw-data directory not found: {source_dir}')
if stage_dir == Path(stage_dir.anchor) or stage_dir == repo_path:
    raise ValueError(f'Unsafe staging target: {stage_dir}')
if stage_dir.exists():
    shutil.rmtree(stage_dir)
stage_dir.mkdir(parents=True)
source_zips = sorted(source_dir.glob('*.zip'), key=lambda path: path.name)
if not source_zips:
    raise FileNotFoundError(f'No ZIP inputs found in {source_dir}')
reference = pd.read_csv(REFERENCE_INVENTORY).sort_values('zip_name', kind='stable')
observed = pd.DataFrame({'zip_name': [path.name for path in source_zips], 'byte_size': [path.stat().st_size for path in source_zips]})
comparison = observed.merge(reference[['zip_name', 'byte_size']], on='zip_name', how='outer', suffixes=('_observed', '_reference'), indicator=True)
mismatch = comparison.loc[(comparison['_merge'] != 'both') | (comparison['byte_size_observed'] != comparison['byte_size_reference'])]
if not mismatch.empty:
    raise RuntimeError(f'Drive/reference inventory mismatch:\n{mismatch.to_string(index=False)}')
copy_evidence = []
for source_path in source_zips:
    target_path = stage_dir / source_path.name
    shutil.copy2(source_path, target_path)
    source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
    target_hash = hashlib.sha256(target_path.read_bytes()).hexdigest()
    if source_path.stat().st_size != target_path.stat().st_size or source_hash != target_hash:
        raise RuntimeError(f'Copy verification failed: {source_path.name}')
    copy_evidence.append((source_path.name, source_path.stat().st_size, source_hash))
print({'staged_zip_count': len(copy_evidence), 'staged_bytes': sum(row[1] for row in copy_evidence), 'sha256_verified': True})

In [ ]:
from src.characterisation.cov_cli import run_cov_characterisation

result = run_cov_characterisation(
    stage_dir,
    Path(OUTPUT_DIR),
    Path(REFERENCE_INVENTORY),
    Path(SITE_CONFIG),
    strict=STRICT_MODE,
)
if STRICT_MODE and result.strict_errors:
    raise RuntimeError(f'Strict COV gate failed after diagnostics: {result.strict_errors}')

In [ ]:
from IPython.display import Markdown, display

summary_frame = pd.DataFrame([result.summary])
display(summary_frame)
display(Markdown(result.artifacts.report_path.read_text(encoding='utf-8')))

In [ ]:
from IPython.display import Image, display

for figure_name in (
    'deadband_instantaneous.png',
    'interarrival_instantaneous.png',
    'canonical_frequency_evidence.png',
    'timestamp_daylight_alignment.png',
    'gap_heartbeat_evidence.png',
):
    display(Image(filename=str(Path(OUTPUT_DIR) / 'figures' / figure_name)))

In [ ]:
if DRIVE_OUTPUT_DIR:
    drive_output = Path(DRIVE_OUTPUT_DIR).resolve()
    drive_output.mkdir(parents=True, exist_ok=True)
    for artifact_path in sorted(Path(OUTPUT_DIR).rglob('*')):
        if artifact_path.is_file():
            relative = artifact_path.relative_to(Path(OUTPUT_DIR))
            destination = drive_output / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(artifact_path, destination)
    print({'drive_output': str(drive_output), 'copied': True})
else:
    print({'drive_output': None, 'copied': False})